In [1]:
import pandas as pd
import numpy as np
import os

# Folder paths
updated_path = r"D:\Swapnil\Work\Projects\FIFA\Updated Base Files"
base_path = r"D:\Swapnil\Work\Projects\FIFA\Base_Files"

# Load Step 5 sales/FIFA file
sales = pd.read_csv(os.path.join(updated_path, "Step5_Global_With_FIFA.csv"))

# Load shopping/customer file from different folder
customers = pd.read_csv(os.path.join(base_path, "shopping_trends_updated.csv"))

print("Sales:", sales.shape)
print("Customers:", customers.shape)
print(customers.columns)

Sales: (1389312, 33)
Customers: (3900, 18)
Index(['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category',
       'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season',
       'Review Rating', 'Subscription Status', 'Shipping Type',
       'Discount Applied', 'Promo Code Used', 'Previous Purchases',
       'Payment Method', 'Frequency of Purchases'],
      dtype='object')


In [2]:
customers.columns = (
    customers.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
)

customers = customers.rename(columns={
    "purchase_amount_usd": "purchase_amount",
    "frequency_of_purchases": "purchase_frequency"
})

print(customers.columns)

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'purchase_frequency'],
      dtype='object')


In [3]:
customer_cols = [
    "customer_id",
    "age",
    "gender",
    "category",
    "purchase_amount",
    "location",
    "season",
    "review_rating",
    "discount_applied",
    "promo_code_used",
    "previous_purchases",
    "payment_method",
    "purchase_frequency"
]

customers = customers[[c for c in customer_cols if c in customers.columns]].copy()

print(customers.head())
print(customers.shape)

   customer_id  age gender  category  purchase_amount       location  season  \
0            1   55   Male  Clothing               53       Kentucky  Winter   
1            2   19   Male  Clothing               64          Maine  Winter   
2            3   50   Male  Clothing               73  Massachusetts  Spring   
3            4   21   Male  Footwear               90   Rhode Island  Spring   
4            5   45   Male  Clothing               49         Oregon  Spring   

   review_rating discount_applied promo_code_used  previous_purchases  \
0            3.1              Yes             Yes                  14   
1            3.1              Yes             Yes                   2   
2            3.1              Yes             Yes                  23   
3            3.5              Yes             Yes                  49   
4            2.7              Yes             Yes                  31   

  payment_method purchase_frequency  
0          Venmo        Fortnightly  
1   

In [4]:
def age_group(age):
    if age < 25:
        return "18-24"
    elif age < 35:
        return "25-34"
    elif age < 45:
        return "35-44"
    elif age < 55:
        return "45-54"
    else:
        return "55+"

customers["age_group"] = customers["age"].apply(age_group)

print(customers[["age", "age_group"]].head())

   age age_group
0   55       55+
1   19     18-24
2   50     45-54
3   21     18-24
4   45     45-54


In [5]:
np.random.seed(42)

sales["customer_id"] = np.random.choice(
    customers["customer_id"],
    size=len(sales),
    replace=True
)

print(sales[["customer_id"]].head())

   customer_id
0         3175
1         3508
2          861
3         1295
4         1131


In [6]:
sales_customers = sales.merge(
    customers,
    on="customer_id",
    how="left"
)

print("After customer join:", sales_customers.shape)

print(sales_customers[[
    "country",
    "company_name",
    "customer_id",
    "age",
    "age_group",
    "gender",
    "purchase_frequency"
]].head())

After customer join: (1389312, 47)
         country company_name  customer_id  age age_group  gender  \
0         Canada    Coca-Cola         3175   45     45-54  Female   
1         Mexico    Coca-Cola         3508   46     45-54  Female   
2  United States    Coca-Cola          861   26     25-34    Male   
3      Argentina    Coca-Cola         1295   27     25-34    Male   
4         Brazil    Coca-Cola         1131   67       55+    Male   

  purchase_frequency  
0            Monthly  
1     Every 3 Months  
2            Monthly  
3     Every 3 Months  
4          Bi-Weekly  


In [7]:
output_path = os.path.join(updated_path, "Step7_Global_With_Customers.csv")

sales_customers.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: D:\Swapnil\Work\Projects\FIFA\Updated Base Files\Step7_Global_With_Customers.csv
